# Validacao Dinamica de Performance no Fabric

Etapa 2 (CD) da esteira: roda **depois** que o Semantic Model chega ao Fabric via Git Integration e passa por um **refresh**.

Cada secao traz, no cabecalho, **o que a analise faz**, **o resultado esperado** e as **faixas de valor** (🟢 bom, 🟡 atencao, 🔴 ruim). Todos os thresholds ficam centralizados no dicionario `CONFIG`.

**Setup (celulas iniciais):** parametros/`CONFIG` e resolucao do dataset por **nome ou id**.

Analises:
1. **Estrutura e memoria** do modelo (model_memory_analyzer): tabelas, colunas, relacionamentos, particoes.
2. **Tamanho e incremental refresh** (tabelas import grandes sem policy).
3. **BPA no modelo publicado** (semantic-link-labs).
4. **Corretude de DAX**: valida se as medidas retornam o valor esperado.
5. **Performance**: mede o tempo medio de queries DAX (N execucoes, cold cache).
7. **Colunas**: cardinalidade, compressao, hierarquias e colunas subutilizadas.
8. **Tabelas e armazenamento**: relacionamentos, segmentos, distribuicao de particoes e tabelas calculadas.
9. **Complexidade das medidas DAX** (score heuristico).
10. **Performance avancada**: percentis p50/p95/p99, variabilidade e carga concorrente.
11. **RLS e refresh**: overhead de seguranca em nivel de linha e tempo de refresh.
12. **Comparacao com baseline historico** (regressao entre runs).
13. **Health score + findings + gate** consolidados (pode virar gate Dev -> Prod).

Pre-requisitos: modelo publicado + com refresh no workspace; ajuste `WORKSPACE`, `DATASET` e os testes de DAX as suas medidas.


In [ ]:
# Rode uma vez se necessario (ja vem no runtime recente do Fabric)
%pip install semantic-link-labs

In [ ]:
import time
import pandas as pd
from datetime import datetime, timezone

import sempy.fabric as fabric
try:
    import sempy_labs as labs
except Exception as e:
    labs = None
    print('sempy_labs indisponivel:', e)

run_ts = datetime.now(timezone.utc)
findings = []

def add_finding(rule, severity, obj_type, obj_name, value=None, threshold=None, details=None):
    findings.append({'run_ts': run_ts, 'rule': rule, 'severity': severity,
                     'object_type': obj_type, 'object_name': obj_name,
                     'value': value, 'threshold': threshold, 'details': details})

def show(df):
    try:
        display(df)
    except Exception:
        print(df)

In [ ]:
import sempy_labs as labs
import sempy.fabric as fabric

workspace = "<workspace_dev>"
datasets_no_workspace = fabric.list_datasets(workspace)
datasets_no_workspace
#labs.admin.list_reports()

In [ ]:
# ===================== PARAMETROS =====================
WORKSPACE = '<workspace_dev>'      # nome ou id do workspace no Fabric
DATASET   = 'CustomerProfitabilitySample'         # nome ou id do Semantic Model a validar

# Performance
PERF_RUNS = 5                    # execucoes por query para medir o tempo
CLEAR_CACHE_BETWEEN = True       # limpa o cache antes de cada execucao (cold cache)

# Limites (health)
CONFIG = {
    # --- Tamanho do modelo (secao 2) ---
    'model_size_mb_attention': 512,
    'model_size_mb_critical': 1024,               # 1 GB -> avaliar incremental
    'large_import_table_data_size_abs': 500000,   # bytes de Data Size
    'large_import_table_data_size_pct_model': 0.10,

    # --- Performance de query (secao 5) ---
    'perf_ms_attention': 1000,
    'perf_ms_critical': 3000,

    # --- Colunas (secao 7): cardinalidade / compressao / hierarquia / subutilizacao ---
    'high_cardinality_attention': 1_000_000,      # valores distintos
    'high_cardinality_critical': 10_000_000,
    'compression_ratio_attention': 0.35,          # Dictionary Size / Data Size (quanto menor melhor)
    'compression_ratio_critical': 0.60,
    'hierarchy_overhead_mb_attention': 30,        # MB gastos com hierarquias de atributo
    'hierarchy_overhead_mb_critical': 80,
    'unused_col_size_mb': 10,                      # coluna grande...
    'unused_col_cardinality': 5,                   # ...com cardinalidade muito baixa

    # --- Tabelas e armazenamento (secao 8) ---
    'relationship_ratio_attention': 100,          # linhas lado-muitos / lado-um
    'relationship_ratio_critical': 1000,
    'rows_per_segment_ok': 1_000_000,             # ideal ~1M linhas por segmento
    'rows_per_segment_attention': 100_000,        # abaixo disso: segmentos pequenos
    'partition_skew_attention': 0.60,             # (max-min)/max entre particoes
    'partition_skew_critical': 0.85,

    # --- Complexidade de medidas DAX (secao 9) ---
    'measure_complexity_attention': 20,
    'measure_complexity_critical': 40,

    # --- Performance avancada (secao 10) ---
    'perf_variance_attention': 0.30,              # (max-min)/media
    'perf_variance_critical': 0.60,
    'perf_p95_multiplier': 1.5,                    # p95 comparado a perf_ms_critical * mult
    'concurrent_users': 5,                         # usuarios simultaneos no teste de carga
    'concurrent_degradation_attention': 0.50,     # +50% no tempo sob carga
    'concurrent_degradation_critical': 1.00,      # +100% (dobrou) sob carga

    # --- RLS (secao 11) ---
    'rls_overhead_attention': 1.5,                 # tempo com RLS / tempo sem RLS
    'rls_overhead_critical': 3.0,

    # --- Comparacao historica (secao 12) ---
    'health_drop_attention': 5,                    # queda no health score vs run anterior
    'health_drop_critical': 10,
}

# Numero de execucoes para calcular percentis (secao 10)
PERF_PERCENTILE_RUNS = 12
RUN_CONCURRENT_TEST = True        # roda o teste de carga concorrente (secao 10)
RLS_TEST_ROLES = []               # ex.: ['RegionalManager']; vazio = pula teste de RLS (secao 11)
REFRESH_TEST_TABLE = None         # ex.: 'Sales'; None = pula teste de refresh (secao 11)

# Testes de corretude (valor esperado das medidas). Ajuste aos seus dados.
DAX_CORRECTNESS_TESTS = [
    {'name': 'Total Revenue', 'dax': 'EVALUATE ROW("v", [Total Revenue])', 'expected': 17250, 'tol': 0.01},
    {'name': 'Total Units',   'dax': 'EVALUATE ROW("v", [Total Units])',   'expected': 700,   'tol': 0.01},
]

# Queries de performance (representam consumo tipico do report)
# Para evitar dependencia de colunas/tabelas especificas, usamos apenas medidas.
PERF_QUERIES = [
    {
        'name': 'Totais (Revenue + Units)',
        'dax': 'EVALUATE ROW("Rev", [Total Revenue], "Units", [Total Units])'
    },
    {
        'name': 'Total Revenue isolado',
        'dax': 'EVALUATE ROW("Rev", [Total Revenue])'
    },
    {
        'name': 'Total Units isolado',
        'dax': 'EVALUATE ROW("Units", [Total Units])'
    },
]

GATE_FAIL_ON = {'High'}          # severidades que barram o gate
WRITE_TO_LAKEHOUSE = False       # True grava findings/summary em Delta (precisa lakehouse anexado)


In [ ]:
def resolve_dataset_id(dataset, workspace):
    try:
        dsets = fabric.list_datasets(workspace)
    except Exception as e:
        print('list_datasets falhou, usando valor informado:', e)
        return dataset
    name_col = next((c for c in dsets.columns if c.lower() in ('dataset name', 'name')), None)
    id_col = next((c for c in dsets.columns if c.lower() in ('dataset id', 'id')), None)
    if id_col and str(dataset) in dsets[id_col].astype(str).values:
        return str(dataset)
    if name_col and id_col:
        m = dsets[dsets[name_col].astype(str) == str(dataset)]
        if len(m):
            return str(m.iloc[0][id_col])
    return dataset

dataset_id = resolve_dataset_id(DATASET, WORKSPACE)
print('Dataset:', DATASET, '->', dataset_id, '| Workspace:', WORKSPACE)

## 1. Estrutura e memoria do modelo

**O que faz:** usa `model_memory_analyzer` para inventariar o modelo publicado — tabelas, colunas, relacionamentos e particoes — com o consumo de memoria (Data Size, Dictionary Size, Hierarchy Size) de cada objeto. E a base (dataframes `tables`, `columns`, `relationships`, `partitions`) usada por quase todas as analises seguintes.

**Resultado:** tabela de resumo do modelo + tabela por objeto com contagem de linhas e tamanho em memoria.

**Faixas de avaliacao:** esta secao e **descritiva** (nao gera classificacao por si so). Ela alimenta os thresholds das secoes 2, 7 e 8. Use para entender o que domina a memoria do modelo.


In [ ]:
mem = fabric.model_memory_analyzer(dataset=dataset_id, workspace=WORKSPACE, return_dataframe=True)

tables = mem['Tables'].copy() if 'Tables' in mem else pd.DataFrame()
columns = mem['Columns'].copy() if 'Columns' in mem else pd.DataFrame()
relationships = mem['Relationships'].copy() if 'Relationships' in mem else pd.DataFrame()
partitions = mem['Partitions'].copy() if 'Partitions' in mem else pd.DataFrame()

if 'Model Summary' in mem:
    show(mem['Model Summary'])
show(tables)

## 2. Tamanho do modelo e incremental refresh

**O que faz:** soma o tamanho total do modelo em memoria e verifica se tabelas **import grandes** possuem politica de **incremental refresh**. Tabela import grande = Data Size >= `large_import_table_data_size_abs` (500 KB) **ou** >= 10% do modelo.

**Resultado:** tamanho total em MB + findings de modelo grande e de tabelas grandes sem incremental.

**Faixas de avaliacao:**

| Metrica | 🟢 Bom | 🟡 Atencao | 🔴 Ruim |
|---|---|---|---|
| Tamanho do modelo | < 512 MB | 512 MB – 1 GB (`MODEL_SIZE_ATTENTION`, Medium) | >= 1 GB (`MODEL_SIZE_CRITICAL`, High) |
| Tabela import grande | com incremental | — | sem incremental (`LARGE_IMPORT_WITHOUT_INCREMENTAL`, High) |


In [ ]:
def to_num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    return df

total_size_mb = 0.0
total_data_size = 0.0
if not tables.empty:
    tables = to_num(tables, ['Row Count', 'Total Size', 'Dictionary Size', 'Data Size'])
    total_size_mb = tables['Total Size'].sum() / 1000000
    total_data_size = tables['Data Size'].sum()
    print('Tamanho total do modelo: {:,.2f} MB'.format(total_size_mb))

    if total_size_mb >= CONFIG['model_size_mb_critical']:
        add_finding('MODEL_SIZE_CRITICAL', 'High', 'Dataset', DATASET, round(total_size_mb, 2), CONFIG['model_size_mb_critical'], 'Modelo grande: avalie incremental/otimizacao')
    elif total_size_mb >= CONFIG['model_size_mb_attention']:
        add_finding('MODEL_SIZE_ATTENTION', 'Medium', 'Dataset', DATASET, round(total_size_mb, 2), CONFIG['model_size_mb_attention'])

# Modo de particao por tabela (Import / DirectQuery / Mixed)
def resolve_modes(part_df):
    out = {}
    if part_df is None or part_df.empty or 'Mode' not in part_df.columns:
        return out
    p = part_df.copy()
    p['Mode'] = p['Mode'].astype(str)
    for tname, grp in p.groupby('Table Name'):
        modes = sorted(set(grp['Mode']))
        out[tname] = modes[0] if len(modes) == 1 else 'Mixed'
    return out

modes = resolve_modes(partitions)
if not tables.empty:
    tables['mode'] = tables['Table Name'].map(modes).fillna('Unknown')

# Incremental refresh apenas em tabela import considerada grande
tom = None
try:
    tom = fabric.TOMWrapper(dataset_id, WORKSPACE, readonly=True)
except Exception as e:
    print('TOMWrapper indisponivel:', e)

if tom is not None and not tables.empty:
    for _, row in tables.iterrows():
        is_import = str(row.get('mode', '')).lower() == 'import'
        data_size = row.get('Data Size', 0)
        is_large = data_size >= CONFIG['large_import_table_data_size_abs'] or (
            total_data_size > 0 and data_size / total_data_size >= CONFIG['large_import_table_data_size_pct_model'])
        if is_import and is_large:
            try:
                if not tom.has_incremental_refresh_policy(table_name=row['Table Name']):
                    add_finding('LARGE_IMPORT_WITHOUT_INCREMENTAL', 'High', 'Table', row['Table Name'], data_size, CONFIG['large_import_table_data_size_abs'], 'Tabela import grande sem incremental refresh')
            except Exception as e:
                add_finding('INCREMENTAL_CHECK_ERROR', 'Low', 'Table', row['Table Name'], details=str(e))

print('OK - checagem de tamanho/incremental concluida')

## 3. BPA no modelo publicado

**O que faz:** roda o **Best Practice Analyzer** (semantic-link-labs) contra o modelo publicado, aplicando o catalogo oficial de regras (performance, formatacao, DAX, manutencao).

**Resultado:** tabela com as violacoes por categoria/severidade. Regras de severidade **Error** viram findings `BPA_ERROR` (High).

**Faixas de avaliacao:**

| Situacao | Classificacao |
|---|---|
| 0 violacoes de erro | 🟢 Bom |
| Violacoes Warning/Info | 🟡 Atencao (revisar, nao bloqueia) |
| Qualquer violacao **Error** | 🔴 Ruim (`BPA_ERROR`, High) |


In [ ]:
bpa = None
if labs is not None:
    try:
        bpa = labs.run_model_bpa(dataset=dataset_id, workspace=WORKSPACE, return_dataframe=True)
        show(bpa)
        sev_col = next((c for c in bpa.columns if c.lower() == 'severity'), None)
        name_col = next((c for c in bpa.columns if c.lower() in ('object name', 'object')), None)
        rule_col = next((c for c in bpa.columns if c.lower() in ('rule name', 'rule')), None)
        if sev_col:
            for _, b in bpa.iterrows():
                sev = str(b.get(sev_col, ''))
                if 'error' in sev.lower() or chr(9940) in sev:
                    add_finding('BPA_ERROR', 'High', 'Model', str(b.get(name_col, '')) if name_col else '', details=str(b.get(rule_col, '')) if rule_col else None)
    except Exception as e:
        print('run_model_bpa falhou:', e)
else:
    print('sempy_labs indisponivel: BPA pulado')

## 4. Corretude de DAX (valores esperados)

**O que faz:** executa cada teste de `DAX_CORRECTNESS_TESTS` e compara o valor retornado com o esperado, dentro de uma tolerancia relativa (`tol`). Garante que a logica das medidas nao quebrou apos o deploy/refresh.

**Resultado:** tabela `expected` x `actual` x `passed` por medida.

**Faixas de avaliacao:**

| Situacao | Classificacao |
|---|---|
| Valor dentro da tolerancia | 🟢 Bom (passed = True) |
| — | 🟡 (nao ha faixa intermediaria: corretude e binaria) |
| Valor fora da tolerancia (`DAX_CORRECTNESS_FAIL`) ou erro na query (`DAX_EVAL_ERROR`) | 🔴 Ruim (High) |


In [ ]:
def eval_scalar(dax):
    df = fabric.evaluate_dax(dataset=dataset_id, dax_string=dax, workspace=WORKSPACE)
    if df is None or len(df) == 0 or df.shape[1] == 0:
        return None
    return df.iloc[0, 0]

corr_rows = []
for t in DAX_CORRECTNESS_TESTS:
    try:
        val = eval_scalar(t['dax'])
        exp = t.get('expected')
        tol = t.get('tol', 0)
        ok = exp is None or (val is not None and abs(float(val) - float(exp)) <= abs(float(exp)) * tol + 1e-9)
        corr_rows.append({'test': t['name'], 'expected': exp, 'actual': val, 'passed': bool(ok)})
        if not ok:
            add_finding('DAX_CORRECTNESS_FAIL', 'High', 'Measure', t['name'], val, exp, 'Valor diferente do esperado')
    except Exception as e:
        corr_rows.append({'test': t['name'], 'expected': t.get('expected'), 'actual': None, 'passed': False})
        add_finding('DAX_EVAL_ERROR', 'High', 'Measure', t['name'], details=str(e))

correctness_df = pd.DataFrame(corr_rows)
show(correctness_df)

## 5. Testes de performance (tempo por query)

**O que faz:** executa cada query de `PERF_QUERIES` `PERF_RUNS` vezes (limpando o cache antes de cada execucao, quando `CLEAR_CACHE_BETWEEN=True`, para medir **cold cache**) e calcula tempo medio/min/max em milissegundos.

**Resultado:** tabela com `avg_ms`, `min_ms`, `max_ms` por query.

**Faixas de avaliacao (tempo medio por query):**

| Faixa | Valor | Classificacao |
|---|---|---|
| Rapido | < 1000 ms | 🟢 Bom |
| Aceitavel/lento | 1000 – 3000 ms (`PERF_ATTENTION`, Medium) | 🟡 Atencao |
| Muito lento | >= 3000 ms (`PERF_CRITICAL`, High) | 🔴 Ruim |


In [ ]:
def clear_cache():
    if labs is None or not CLEAR_CACHE_BETWEEN:
        return
    try:
        labs.clear_cache(dataset=dataset_id, workspace=WORKSPACE)
    except Exception:
        pass

perf_rows = []
for q in PERF_QUERIES:
    durs = []
    for _ in range(PERF_RUNS):
        clear_cache()
        t0 = time.perf_counter()
        try:
            _ = fabric.evaluate_dax(dataset=dataset_id, dax_string=q['dax'], workspace=WORKSPACE)
            durs.append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            add_finding('PERF_QUERY_ERROR', 'Medium', 'Query', q['name'], details=str(e))
            durs = []
            break
    if durs:
        avg = sum(durs) / len(durs)
        perf_rows.append({'query': q['name'], 'runs': len(durs), 'avg_ms': round(avg, 1),
                          'min_ms': round(min(durs), 1), 'max_ms': round(max(durs), 1)})
        if avg >= CONFIG['perf_ms_critical']:
            add_finding('PERF_CRITICAL', 'High', 'Query', q['name'], round(avg, 1), CONFIG['perf_ms_critical'])
        elif avg >= CONFIG['perf_ms_attention']:
            add_finding('PERF_ATTENTION', 'Medium', 'Query', q['name'], round(avg, 1), CONFIG['perf_ms_attention'])

perf_df = pd.DataFrame(perf_rows)
show(perf_df)

# Resumo textual em ms para facilitar leitura no output
if perf_rows:
    print("\nResumo de performance (ms):")
    for r in perf_rows:
        print(f"- {r['query']}: avg={r['avg_ms']} ms, min={r['min_ms']} ms, max={r['max_ms']} ms, runs={r['runs']}")
else:
    print("Nenhum resultado de performance calculado (verifique se as queries DAX estao validas em PERF_QUERIES).")

## 7. Analise de colunas (cardinalidade, compressao, hierarquias, subutilizacao)

**O que faz:** aprofunda a memoria por coluna. (a) **Cardinalidade** — n. de valores distintos, o maior fator de tamanho/velocidade no motor VertiPaq; (b) **Compressao** por tabela = `Dictionary Size / Data Size` (quanto menor, melhor a compressao); (c) **Hierarquias** — memoria gasta com hierarquias de atributo (colunas raramente filtradas nao precisam delas); (d) **Colunas subutilizadas** — colunas grandes com cardinalidade baixissima, candidatas a remocao.

**Resultado:** top 15 colunas por cardinalidade + tabela de compressao por tabela + overhead total de hierarquias.

**Faixas de avaliacao:**

| Metrica | 🟢 Bom | 🟡 Atencao | 🔴 Ruim |
|---|---|---|---|
| Cardinalidade da coluna | < 1M | 1M – 10M (Medium) | >= 10M (High) |
| Razao dict/data (compressao) | < 0.35 | 0.35 – 0.59 (Low) | >= 0.60 (Medium) |
| Overhead de hierarquias | < 30 MB | 30 – 80 MB (Low) | >= 80 MB (Medium) |
| Coluna subutilizada | — | size >= 10 MB e cardinalidade <= 5 (Low) | — |


In [ ]:
# ===== Analise de colunas: cardinalidade, compressao, hierarquias e subutilizacao =====
if not columns.empty:
    columns = to_num(columns, ['Cardinality', 'Data Size', 'Dictionary Size', 'Hierarchy Size', 'Total Size'])

    # 1) Cardinalidade (valores distintos) — principal fator de tamanho/velocidade
    for _, col in columns.iterrows():
        card = col.get('Cardinality', 0)
        obj = '{}.{}'.format(col.get('Table Name', '?'), col.get('Column Name', '?'))
        if card >= CONFIG['high_cardinality_critical']:
            add_finding('HIGH_CARDINALITY', 'High', 'Column', obj, int(card),
                        CONFIG['high_cardinality_critical'], 'Cardinalidade muito alta')
        elif card >= CONFIG['high_cardinality_attention']:
            add_finding('HIGH_CARDINALITY', 'Medium', 'Column', obj, int(card),
                        CONFIG['high_cardinality_attention'], 'Cardinalidade alta')

    # 2) Compressao por tabela (Dictionary Size / Data Size)
    comp_rows = []
    for tname, grp in columns.groupby('Table Name'):
        dict_sz = grp['Dictionary Size'].sum()
        data_sz = grp['Data Size'].sum()
        ratio = (dict_sz / data_sz) if data_sz > 0 else 0
        comp_rows.append({'table': tname, 'dict_size': int(dict_sz), 'data_size': int(data_sz),
                          'dict_data_ratio': round(ratio, 3)})
        if ratio >= CONFIG['compression_ratio_critical']:
            add_finding('POOR_COMPRESSION', 'Medium', 'Table', tname, round(ratio, 3),
                        CONFIG['compression_ratio_critical'], 'Compressao ruim')
        elif ratio >= CONFIG['compression_ratio_attention']:
            add_finding('POOR_COMPRESSION', 'Low', 'Table', tname, round(ratio, 3),
                        CONFIG['compression_ratio_attention'], 'Compressao moderada')
    compression_df = pd.DataFrame(comp_rows).sort_values('dict_data_ratio', ascending=False) if comp_rows else pd.DataFrame()

    # 3) Overhead de hierarquias de atributo
    hier_total_mb = columns['Hierarchy Size'].sum() / 1_000_000
    if hier_total_mb >= CONFIG['hierarchy_overhead_mb_critical']:
        add_finding('LARGE_HIERARCHY_OVERHEAD', 'Medium', 'Model', DATASET, round(hier_total_mb, 2),
                    CONFIG['hierarchy_overhead_mb_critical'], 'Overhead de hierarquias alto')
    elif hier_total_mb >= CONFIG['hierarchy_overhead_mb_attention']:
        add_finding('LARGE_HIERARCHY_OVERHEAD', 'Low', 'Model', DATASET, round(hier_total_mb, 2),
                    CONFIG['hierarchy_overhead_mb_attention'], 'Overhead de hierarquias moderado')

    # 4) Colunas potencialmente subutilizadas (grandes + cardinalidade muito baixa)
    for _, col in columns.iterrows():
        size_mb = col.get('Total Size', 0) / 1_000_000
        card = col.get('Cardinality', 1)
        if size_mb >= CONFIG['unused_col_size_mb'] and card <= CONFIG['unused_col_cardinality']:
            obj = '{}.{}'.format(col.get('Table Name', '?'), col.get('Column Name', '?'))
            add_finding('POTENTIALLY_UNUSED_COLUMN', 'Low', 'Column', obj, round(size_mb, 2),
                        CONFIG['unused_col_size_mb'], 'Coluna grande com baixa cardinalidade')

    cols_view = columns.copy()
    cols_view['size_mb'] = (cols_view['Total Size'] / 1_000_000).round(2)
    cardinality_df = cols_view[['Table Name', 'Column Name', 'Cardinality', 'size_mb']].sort_values(
        'Cardinality', ascending=False).head(15)

    print('Overhead total de hierarquias: {:.2f} MB'.format(hier_total_mb))
    show(cardinality_df)
    show(compression_df)
else:
    cardinality_df = pd.DataFrame()
    compression_df = pd.DataFrame()
    print('columns vazio: analise de colunas pulada')


## 8. Tabelas e armazenamento

**O que faz:** avalia a "saude fisica" das tabelas. (a) **Relacionamentos** — proporcao de linhas entre o lado-muitos e o lado-um (proporcoes altissimas encarecem a materializacao no motor); (b) **Segmentos do columnstore** — linhas por segmento (segmentos muito pequenos pioram a compressao e o scan); (c) **Distribuicao de particoes** — skew `(max-min)/max` (particoes desbalanceadas prejudicam refresh paralelo); (d) **Tabelas calculadas** — sinaliza tabelas que talvez devessem vir do ETL.

**Resultado:** tabela de relacionamentos, tabela de segmentos e lista de tabelas calculadas.

**Faixas de avaliacao:**

| Metrica | 🟢 Bom | 🟡 Atencao | 🔴 Ruim |
|---|---|---|---|
| Proporcao do relacionamento | < 100x | 100x – 999x (Low) | >= 1000x (Medium) |
| Linhas por segmento | >= 1.000.000 | 100.000 – 999.999 | < 100.000 (`SMALL_SEGMENTS`, Low) |
| Skew de particoes | < 0.60 | 0.60 – 0.84 (Low) | >= 0.85 (Medium) |
| Tabela calculada | nenhuma | existe (Low, revisar) | — |


In [ ]:
# ===== Analise de tabelas e armazenamento =====
# 1) Qualidade dos relacionamentos (proporcao linhas lado-muitos / lado-um)
rel_rows = []
if not relationships.empty and not tables.empty:
    # dicionario: nome da tabela -> row count
    table_rows = {str(r['Table Name']).strip(): r.get('Row Count', 0) for _, r in tables.iterrows()}

    # Nas saídas atuais do model_memory_analyzer, os relacionamentos vêm como
    # 'From Object' = "'Fact'[Customer Key]" e 'To Object' = "'Customer'[Customer]".
    # Precisamos extrair apenas o nome da tabela (entre aspas simples, antes do colchete).
    from_col = 'From Object'
    to_col = 'To Object'

    def _extract_table_name(obj_str: str):
        if not isinstance(obj_str, str):
            obj_str = str(obj_str)
        s = obj_str.strip()
        # Formato tipico: 'Table'[Column]
        if s.startswith("'") and "'" in s[1:]:
            try:
                # pega o que está entre a primeira e a segunda aspa simples
                inner = s.split("'", 2)[1]
                return inner.strip()
            except Exception:
                return s
        # fallback: se nao tiver aspas, tenta antes do colchete
        if '[' in s:
            return s.split('[', 1)[0].strip()
        return s

    for _, rel in relationships.iterrows():
        ft_raw = rel.get(from_col)
        tt_raw = rel.get(to_col)
        ft = _extract_table_name(ft_raw)
        tt = _extract_table_name(tt_raw)
        fr = table_rows.get(ft, 0)
        tr = table_rows.get(tt, 0)
        ratio = (fr / tr) if tr > 0 else 0
        rel_rows.append({'from': ft, 'to': tt, 'from_rows': fr, 'to_rows': tr, 'ratio': round(ratio, 1)})
        if ratio >= CONFIG['relationship_ratio_critical']:
            add_finding('RELATIONSHIP_CARDINALITY', 'Medium', 'Relationship', f'{ft} -> {tt}',
                        round(ratio, 1), CONFIG['relationship_ratio_critical'], 'Proporcao muito alta')
        elif ratio >= CONFIG['relationship_ratio_attention']:
            add_finding('RELATIONSHIP_CARDINALITY', 'Low', 'Relationship', f'{ft} -> {tt}',
                        round(ratio, 1), CONFIG['relationship_ratio_attention'], 'Proporcao alta')
relationships_quality_df = pd.DataFrame(rel_rows)

# 2) Segmentos do columnstore + 3) distribuicao (skew) de particoes
seg_rows = []
if not partitions.empty:
    # Nas partitions deste modelo, os campos sao 'Record Count' e 'Segment Count'
    partitions = to_num(partitions, ['Record Count', 'Segment Count'])
    if 'Segment Count' in partitions.columns and 'Record Count' in partitions.columns:
        for _, p in partitions.iterrows():
            segc = p.get('Segment Count', 0)
            rc = p.get('Record Count', 0)
            if segc > 0 and rc > 0:
                rps = rc / segc
                seg_rows.append({'table': p.get('Table Name'), 'partition': p.get('Partition Name', ''),
                                 'rows': int(rc), 'segments': int(segc), 'rows_per_segment': int(rps)})
                if rps < CONFIG['rows_per_segment_attention']:
                    add_finding('SMALL_SEGMENTS', 'Low', 'Partition', str(p.get('Table Name')),
                                int(rps), CONFIG['rows_per_segment_attention'], 'Segmentos pequenos')
    # skew de particoes por tabela
    if 'Record Count' in partitions.columns:
        for tname, grp in partitions.groupby('Table Name'):
            if len(grp) > 1 and grp['Record Count'].max() > 0:
                rc = grp['Record Count'].values
                skew = (rc.max() - rc.min()) / rc.max()
                if skew >= CONFIG['partition_skew_critical']:
                    add_finding('PARTITION_SKEW', 'Medium', 'Table', tname, round(float(skew), 2),
                                CONFIG['partition_skew_critical'], 'Particoes muito desbalanceadas')
                elif skew >= CONFIG['partition_skew_attention']:
                    add_finding('PARTITION_SKEW', 'Low', 'Table', tname, round(float(skew), 2),
                                CONFIG['partition_skew_attention'], 'Particoes desbalanceadas')
segments_df = pd.DataFrame(seg_rows)

# 4) Tabelas calculadas (candidatas a virem direto da fonte)
calc_tables = []
if tom is not None:
    try:
        for table in tom.model.Tables:
            for partition in table.Partitions:
                if 'Calculated' in str(getattr(partition, 'SourceType', '')):
                    calc_tables.append(table.Name)
                    add_finding('CALCULATED_TABLE_REVIEW', 'Low', 'Table', table.Name,
                                details='Revisar se pode vir direto da fonte (ETL)')
                    break
    except Exception as e:
        print('Analise de calculated tables:', e)

show(relationships_quality_df)
show(segments_df)
print('Tabelas calculadas:', calc_tables if calc_tables else 'nenhuma')


## 9. Complexidade das medidas DAX

**O que faz:** calcula um **score heuristico de complexidade** para cada medida a partir da expressao DAX, ponderando funcoes custosas e iteradoras (`CALCULATE` x3, `RANKX` x3, `FILTER`/`SUMX`/`AVERAGEX` x2, `RELATED` x1) mais o tamanho do texto. Medidas muito complexas tendem a ser mais lentas e dificeis de manter.

**Resultado:** tabela das medidas ordenadas por complexidade (top 15).

**Faixas de avaliacao (score de complexidade):**

| Faixa | Valor | Classificacao |
|---|---|---|
| Simples | < 20 | 🟢 Bom |
| Complexa | 20 – 39 (`COMPLEX_MEASURE`, Low) | 🟡 Atencao |
| Muito complexa | >= 40 (`COMPLEX_MEASURE`, Medium) | 🔴 Ruim |


In [ ]:
# ===== Complexidade das medidas DAX =====
# Score heuristico: pondera funcoes custosas/iteradoras e o tamanho da expressao.
measure_complexity_df = pd.DataFrame()
try:
    measures = fabric.list_measures(dataset_id, WORKSPACE)
    expr_col = next((c for c in measures.columns if 'expression' in c.lower()), None)
    name_col = next((c for c in measures.columns if c.lower() in ('measure name', 'measure', 'name')), None)
    rows = []
    for _, m in measures.iterrows():
        dax = str(m.get(expr_col, '')) if expr_col else ''
        up = dax.upper()
        score = (up.count('CALCULATE') * 3 + up.count('FILTER') * 2 + up.count('SUMX') * 2 +
                 up.count('AVERAGEX') * 2 + up.count('RANKX') * 3 + up.count('RELATED') * 1 +
                 len(dax) / 100)
        name = m.get(name_col, '') if name_col else ''
        rows.append({'measure': name, 'complexity': round(score, 1), 'length': len(dax)})
        if score >= CONFIG['measure_complexity_critical']:
            add_finding('COMPLEX_MEASURE', 'Medium', 'Measure', name, round(score, 1),
                        CONFIG['measure_complexity_critical'], 'Medida muito complexa')
        elif score >= CONFIG['measure_complexity_attention']:
            add_finding('COMPLEX_MEASURE', 'Low', 'Measure', name, round(score, 1),
                        CONFIG['measure_complexity_attention'], 'Medida complexa')
    measure_complexity_df = pd.DataFrame(rows).sort_values('complexity', ascending=False) if rows else pd.DataFrame()
    show(measure_complexity_df.head(15))
except Exception as e:
    print('list_measures indisponivel: analise de complexidade pulada:', e)


## 10. Performance avancada: percentis, variabilidade e carga concorrente

**O que faz:** vai alem da media da secao 5. (a) Reexecuta cada query `PERF_PERCENTILE_RUNS` vezes e calcula **p50/p95/p99** (o p95 representa a experiencia do "pior caso tipico" que 95% dos usuarios veem); (b) mede a **variabilidade** `(max-min)/media` — inconsistencia indica contencao de recursos ou cache instavel; (c) faz um **teste de carga concorrente** disparando a query mais pesada com varios usuarios simultaneos e mede a degradacao.

**Resultado:** tabela de percentis + variancia por query e tabela de degradacao sob carga.

**Faixas de avaliacao:**

| Metrica | 🟢 Bom | 🟡 Atencao | 🔴 Ruim |
|---|---|---|---|
| p95 | < 4500 ms | — | >= 4500 ms (3000 × 1.5) → `P95_PERFORMANCE` (High) |
| Variabilidade | < 0.30 | 0.30 – 0.60 (Low) | >= 0.60 (Medium) |
| Degradacao sob carga | < 50% | 50% – 99% (Medium) | >= 100% / dobrou (High) |


In [ ]:
# ===== Performance avancada: percentis (p50/p95/p99), variabilidade e carga concorrente =====
import numpy as np
import concurrent.futures

# Percentis + variabilidade: reexecuta cada query PERF_PERCENTILE_RUNS vezes (cold cache).
perc_rows = []
for q in PERF_QUERIES:
    durs = []
    for _ in range(PERF_PERCENTILE_RUNS):
        clear_cache()
        t0 = time.perf_counter()
        try:
            fabric.evaluate_dax(dataset=dataset_id, dax_string=q['dax'], workspace=WORKSPACE)
            durs.append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            add_finding('PERF_QUERY_ERROR', 'Medium', 'Query', q['name'], details=str(e))
            durs = []
            break
    if len(durs) >= 3:
        p50 = float(np.percentile(durs, 50))
        p95 = float(np.percentile(durs, 95))
        p99 = float(np.percentile(durs, 99))
        avg = sum(durs) / len(durs)
        variance = (max(durs) - min(durs)) / avg if avg > 0 else 0
        perc_rows.append({'query': q['name'], 'samples': len(durs),
                          'p50_ms': round(p50, 1), 'p95_ms': round(p95, 1),
                          'p99_ms': round(p99, 1), 'variance': round(variance, 2)})
        p95_limit = CONFIG['perf_ms_critical'] * CONFIG['perf_p95_multiplier']
        if p95 >= p95_limit:
            add_finding('P95_PERFORMANCE', 'High', 'Query', q['name'], round(p95, 1),
                        round(p95_limit, 1), 'P95 acima do limite (95% das execucoes)')
        if variance >= CONFIG['perf_variance_critical']:
            add_finding('PERFORMANCE_INCONSISTENCY', 'Medium', 'Query', q['name'],
                        round(variance, 2), CONFIG['perf_variance_critical'], 'Alta variabilidade')
        elif variance >= CONFIG['perf_variance_attention']:
            add_finding('PERFORMANCE_INCONSISTENCY', 'Low', 'Query', q['name'],
                        round(variance, 2), CONFIG['perf_variance_attention'])
perf_percentiles_df = pd.DataFrame(perc_rows)
show(perf_percentiles_df)

# Carga concorrente: dispara a query mais pesada com N usuarios simultaneos e mede a degradacao.
concurrent_df = pd.DataFrame()
if RUN_CONCURRENT_TEST and 'perf_rows' in dir() and perf_rows:
    heaviest = max(perf_rows, key=lambda x: x['avg_ms'])
    hq = next((q['dax'] for q in PERF_QUERIES if q['name'] == heaviest['query']), None)

    def _single_query():
        t0 = time.perf_counter()
        try:
            fabric.evaluate_dax(dataset=dataset_id, dax_string=hq, workspace=WORKSPACE)
            return (time.perf_counter() - t0) * 1000
        except Exception:
            return None

    n = CONFIG['concurrent_users']
    with concurrent.futures.ThreadPoolExecutor(max_workers=n) as ex:
        results = [f.result() for f in [ex.submit(_single_query) for _ in range(n * 2)]]
    results = [r for r in results if r is not None]
    if results:
        cavg = sum(results) / len(results)
        degr = (cavg / heaviest['avg_ms']) - 1 if heaviest['avg_ms'] > 0 else 0
        concurrent_df = pd.DataFrame([{'query': heaviest['query'], 'users': n,
                                       'single_avg_ms': heaviest['avg_ms'],
                                       'concurrent_avg_ms': round(cavg, 1),
                                       'degradation_pct': round(degr * 100, 1)}])
        if degr >= CONFIG['concurrent_degradation_critical']:
            add_finding('CONCURRENT_DEGRADATION', 'High', 'Query', heaviest['query'],
                        round(degr * 100, 1), CONFIG['concurrent_degradation_critical'] * 100,
                        'Performance degrada muito sob carga')
        elif degr >= CONFIG['concurrent_degradation_attention']:
            add_finding('CONCURRENT_DEGRADATION', 'Medium', 'Query', heaviest['query'],
                        round(degr * 100, 1), CONFIG['concurrent_degradation_attention'] * 100)
        show(concurrent_df)
else:
    print('Teste concorrente pulado (RUN_CONCURRENT_TEST=False ou sem perf_rows da secao 5).')


## 11. Impacto de RLS e tempo de refresh

**O que faz:** (a) **RLS** — executa a mesma query com e sem impersonation de role e mede o overhead que a seguranca em nivel de linha adiciona; (b) **Refresh** — mede o tempo de um refresh de tabela (opcional, custoso). Ambos sao **opt-in**: RLS roda so se `RLS_TEST_ROLES` estiver preenchido e refresh so se `REFRESH_TEST_TABLE` for definido.

**Resultado:** tabela de overhead por role e tabela com o tempo de refresh.

**Faixas de avaliacao (overhead de RLS = tempo_com_rls / tempo_sem_rls):**

| Faixa | Valor | Classificacao |
|---|---|---|
| Baixo | < 1.5x | 🟢 Bom |
| Moderado | 1.5x – 3x (Low) | 🟡 Atencao |
| Alto | >= 3x (Medium) | 🔴 Ruim |

> O tempo de refresh e **informativo** (`REFRESH_MEASURED`), sem faixa fixa — compare com a janela de refresh do seu SLA.


In [ ]:
# ===== Impacto de RLS na performance e tempo de refresh =====
# RLS: compara o tempo da mesma query com e sem impersonation de role.
rls_df = pd.DataFrame()
if RLS_TEST_ROLES and labs is not None:
    base_q = PERF_QUERIES[0]['dax'] if PERF_QUERIES else 'EVALUATE ROW("x", 1)'
    rls_rows = []
    for role in RLS_TEST_ROLES:
        try:
            clear_cache(); t0 = time.perf_counter()
            fabric.evaluate_dax(dataset=dataset_id, dax_string=base_q, workspace=WORKSPACE)
            base_ms = (time.perf_counter() - t0) * 1000

            clear_cache(); t0 = time.perf_counter()
            # Depende da versao do sempy-labs; se nao existir, o teste e pulado com aviso.
            labs.evaluate_dax_impersonation(dataset=dataset_id, dax_string=base_q,
                                            workspace=WORKSPACE, user_name=role)
            rls_ms = (time.perf_counter() - t0) * 1000

            overhead = rls_ms / base_ms if base_ms > 0 else 0
            rls_rows.append({'role': role, 'base_ms': round(base_ms, 1),
                             'rls_ms': round(rls_ms, 1), 'overhead_x': round(overhead, 2)})
            if overhead >= CONFIG['rls_overhead_critical']:
                add_finding('RLS_PERFORMANCE_IMPACT', 'Medium', 'Security', role,
                            round(overhead, 2), CONFIG['rls_overhead_critical'], 'RLS muito custoso')
            elif overhead >= CONFIG['rls_overhead_attention']:
                add_finding('RLS_PERFORMANCE_IMPACT', 'Low', 'Security', role,
                            round(overhead, 2), CONFIG['rls_overhead_attention'])
        except Exception as e:
            add_finding('RLS_TEST_ERROR', 'Low', 'Security', role, details=str(e))
            print('Teste de RLS nao suportado para role {}: {}'.format(role, e))
    rls_df = pd.DataFrame(rls_rows)
    show(rls_df)
else:
    print('RLS_TEST_ROLES vazio: teste de RLS pulado.')

# Refresh: mede o tempo de refresh de uma tabela (custoso; use apenas em ambiente de teste).
refresh_df = pd.DataFrame()
if REFRESH_TEST_TABLE and labs is not None:
    try:
        t0 = time.perf_counter()
        labs.refresh_semantic_model(dataset=dataset_id, workspace=WORKSPACE,
                                    tables=[REFRESH_TEST_TABLE], refresh_type='full')
        dur = (time.perf_counter() - t0) * 1000
        refresh_df = pd.DataFrame([{'table': REFRESH_TEST_TABLE, 'refresh_ms': round(dur, 1)}])
        add_finding('REFRESH_MEASURED', 'Low', 'Table', REFRESH_TEST_TABLE, round(dur, 1),
                    None, 'Tempo de refresh medido (informativo)')
        show(refresh_df)
    except Exception as e:
        print('Refresh nao executado:', e)
else:
    print('REFRESH_TEST_TABLE=None: teste de refresh pulado.')


## 12. Comparacao com baseline historico

**O que faz:** le os ultimos runs salvos na tabela `dyn_val_health_summary` (lakehouse) e compara o **health score provisorio** deste run com o do run anterior, detectando regressoes de qualidade/performance ao longo do tempo. So funciona com `WRITE_TO_LAKEHOUSE=True` e apos existir historico.

**Resultado:** tabela com os ultimos scores + finding `HEALTH_DEGRADATION` quando houver queda relevante.

**Faixas de avaliacao (queda de pontos vs. run anterior):**

| Faixa | Valor | Classificacao |
|---|---|---|
| Estavel ou melhorou | queda < 5 pts | 🟢 Bom |
| Regressao leve | queda 5 – 9 pts (Medium) | 🟡 Atencao |
| Regressao forte | queda >= 10 pts (High) | 🔴 Ruim |


In [ ]:
# ===== Comparacao com baseline historico =====
# Compara o score provisorio deste run com o ultimo run salvo no lakehouse.
history_df = pd.DataFrame()
_weights_hist = {'High': 15, 'Medium': 7, 'Low': 2}
prov_score = max(0, min(100, 100 - sum(_weights_hist.get(f['severity'], 5) for f in findings)))

if WRITE_TO_LAKEHOUSE:
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        hist = spark.sql(
            "SELECT run_ts, health_score FROM dyn_val_health_summary "
            "WHERE dataset = '{}' ORDER BY run_ts DESC LIMIT 5".format(DATASET)
        ).toPandas()
        history_df = hist
        if len(hist) >= 1:
            prev = float(hist.iloc[0]['health_score'])
            drop = prev - prov_score
            print('Score anterior: {} | provisorio atual: {} | variacao: {:+.0f}'.format(prev, prov_score, -drop))
            if drop >= CONFIG['health_drop_critical']:
                add_finding('HEALTH_DEGRADATION', 'High', 'Model', DATASET, prov_score, prev,
                            'Health caiu {:.0f} pontos vs ultimo run'.format(drop))
            elif drop >= CONFIG['health_drop_attention']:
                add_finding('HEALTH_DEGRADATION', 'Medium', 'Model', DATASET, prov_score, prev,
                            'Health caiu {:.0f} pontos vs ultimo run'.format(drop))
        show(history_df)
    except Exception as e:
        print('Comparacao historica indisponivel (ainda sem historico?):', e)
else:
    print('WRITE_TO_LAKEHOUSE=False: comparacao historica pulada (score provisorio = {}).'.format(prov_score))


## 13. Health score, findings e gate

**O que faz:** consolida **todos** os findings gerados nas secoes anteriores em um score unico (0–100). Cada finding desconta pontos conforme a severidade: **High = -15**, **Medium = -7**, **Low = -2**. Tambem monta o resumo (`summary`) e a lista detalhada (`findings_df`).

**Resultado:** health score + status textual + tabelas de resumo e findings.

**Faixas de avaliacao (health score):**

| Faixa | Valor | Status |
|---|---|---|
| Saudavel | >= 90 | 🟢 Bom |
| Atencao | 75 – 89 | 🟡 Atencao |
| Ruim | 50 – 74 | 🟠 Ruim |
| Critico | < 50 | 🔴 Critico |

O **gate** (celula seguinte) bloqueia a promocao Dev -> Prod se houver qualquer finding com severidade em `GATE_FAIL_ON` (por padrao, **High**).


In [ ]:
weights = {'High': 15, 'Medium': 7, 'Low': 2}
score = 100
for f in findings:
    score -= weights.get(f['severity'], 5)
score = max(0, min(100, score))
status = 'Saudavel' if score >= 90 else 'Atencao' if score >= 75 else 'Ruim' if score >= 50 else 'Critico'

corr_pass = int(correctness_df['passed'].sum()) if len(correctness_df) else 0
worst_perf = float(perf_df['avg_ms'].max()) if len(perf_df) else None

summary = pd.DataFrame([{
    'run_ts': run_ts, 'dataset': DATASET, 'workspace': WORKSPACE,
    'model_size_mb': round(total_size_mb, 2),
    'correctness_passed': corr_pass, 'correctness_total': len(correctness_df),
    'perf_worst_avg_ms': worst_perf,
    'findings': len(findings), 'health_score': score, 'health_status': status,
}])
findings_df = pd.DataFrame(findings)

print('Health: {} ({})'.format(score, status))
show(summary)
show(findings_df)

In [ ]:
# (Opcional) Grava resultados em Delta no lakehouse anexado, para historico/monitoramento
if WRITE_TO_LAKEHOUSE:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    def norm(df):
        return df.toDF(*[c.strip().lower().replace(' ', '_') for c in df.columns])

    def _safe(name):
        return globals().get(name, None)

    datasets_to_save = [
        ('dyn_val_perf_results', perf_df),
        ('dyn_val_dax_correctness', correctness_df),
        ('dyn_val_health_summary', summary),
        ('dyn_val_health_findings', findings_df),
        # Novas analises
        ('dyn_val_cardinality', _safe('cardinality_df')),
        ('dyn_val_compression', _safe('compression_df')),
        ('dyn_val_relationships_quality', _safe('relationships_quality_df')),
        ('dyn_val_segments', _safe('segments_df')),
        ('dyn_val_measure_complexity', _safe('measure_complexity_df')),
        ('dyn_val_perf_percentiles', _safe('perf_percentiles_df')),
        ('dyn_val_concurrent', _safe('concurrent_df')),
        ('dyn_val_rls', _safe('rls_df')),
        ('dyn_val_refresh', _safe('refresh_df')),
    ]
    for name, pdf in datasets_to_save:
        if pdf is not None and len(pdf):
            sdf = norm(spark.createDataFrame(pdf.astype(str)))
            sdf.write.mode('append').format('delta').saveAsTable(name)
    print('Gravado no lakehouse anexado.')
else:
    print('WRITE_TO_LAKEHOUSE = False: nada gravado (apenas exibicao).')


## 14. Consolidação final dos resultados (itens 1 a 13)

In [ ]:
# ===== 14. Consolidação final dos resultados (itens 1 a 13) =====

# Item 1 - Estrutura e memória do modelo
print("=== Item 1 - Estrutura e memória do modelo ===")
try:
    print("Tabelas do modelo (tamanho/memória):")
    show(tables)
except Exception as e:
    print("tables indisponível:", e)

# Item 2 - Tamanho do modelo e incremental refresh
print("\n=== Item 2 - Tamanho do modelo e incremental refresh ===")
try:
    print(f"Tamanho total do modelo: {total_size_mb:,.2f} MB")
    if 'findings_df' in globals() and len(findings_df):
        f2 = findings_df[findings_df['rule'].isin([
            'MODEL_SIZE_CRITICAL', 'MODEL_SIZE_ATTENTION', 'LARGE_IMPORT_WITHOUT_INCREMENTAL',
            'INCREMENTAL_CHECK_ERROR'
        ])]
        if len(f2):
            print("Findings relacionados ao tamanho/incremental:")
            show(f2)
        else:
            print("Sem findings específicos de tamanho/incremental.")
    else:
        print("Nenhum finding calculado ainda.")
except Exception as e:
    print("Resumo de tamanho/incremental indisponível:", e)

# Item 3 - BPA no modelo publicado
print("\n=== Item 3 - BPA no modelo publicado ===")
try:
    if 'bpa' in globals() and bpa is not None and len(bpa):
        show(bpa)
    else:
        print("BPA não executado ou sem resultados.")
except Exception as e:
    print("bpa indisponível:", e)

# Item 4 - Corretude de DAX
print("\n=== Item 4 - Corretude de DAX ===")
try:
    show(correctness_df)
except Exception as e:
    print("correctness_df indisponível:", e)

# Item 5 - Performance básica (tempo médio por query)
print("\n=== Item 5 - Performance básica (tempo médio por query) ===")
try:
    show(perf_df)
except Exception as e:
    print("perf_df indisponível:", e)

# (Item 6 não existe no roteiro atual)

# Item 7 - Colunas (cardinalidade e compressão)
print("\n=== Item 7 - Colunas (cardinalidade e compressão) ===")
try:
    show(cardinality_df)
except Exception as e:
    print("cardinality_df indisponível:", e)

try:
    show(compression_df)
except Exception as e:
    print("compression_df indisponível:", e)

# Item 8 - Tabelas e armazenamento (relacionamentos, segmentos)
print("\n=== Item 8 - Tabelas e armazenamento (relacionamentos, segmentos) ===")
try:
    show(relationships_quality_df)
except Exception as e:
    print("relationships_quality_df indisponível:", e)

try:
    show(segments_df)
except Exception as e:
    print("segments_df indisponível:", e)

# Item 9 - Complexidade das medidas DAX
print("\n=== Item 9 - Complexidade das medidas DAX ===")
try:
    show(measure_complexity_df.head(20))
except Exception as e:
    print("measure_complexity_df indisponível:", e)

# Item 10 - Performance avançada (percentis e carga concorrente)
print("\n=== Item 10 - Performance avançada (percentis e carga concorrente) ===")
try:
    show(perf_percentiles_df)
except Exception as e:
    print("perf_percentiles_df indisponível:", e)

try:
    if 'concurrent_df' in globals() and len(concurrent_df):
        show(concurrent_df)
    else:
        print("Teste concorrente não executado ou sem resultados.")
except Exception as e:
    print("concurrent_df indisponível:", e)

# Item 11 - RLS e refresh
print("\n=== Item 11 - RLS e refresh ===")
try:
    if 'rls_df' in globals() and len(rls_df):
        show(rls_df)
    else:
        print("RLS_TEST_ROLES vazio ou sem resultados.")
except Exception as e:
    print("rls_df indisponível:", e)

try:
    if 'refresh_df' in globals() and len(refresh_df):
        show(refresh_df)
    else:
        print("REFRESH_TEST_TABLE=None ou sem resultados de refresh.")
except Exception as e:
    print("refresh_df indisponível:", e)

# Item 12 - Histórico de health
print("\n=== Item 12 - Histórico de health (quando WRITE_TO_LAKEHOUSE=True) ===")
try:
    if 'history_df' in globals() and len(history_df):
        show(history_df)
    else:
        print("WRITE_TO_LAKEHOUSE=False ou ainda sem histórico.")
except Exception as e:
    print("history_df indisponível:", e)

# Item 13 - Health score, findings e gate
print("\n=== Item 13 - Health score, findings e gate ===")
try:
    print("Health geral:")
    show(summary)
except Exception as e:
    print("summary indisponível:", e)

print("\nFindings consolidados (ordenados por severidade e regra):")
try:
    if 'findings_df' in globals() and len(findings_df):
        findings_sorted = findings_df.sort_values(['severity', 'rule', 'object_type', 'object_name'])
        show(findings_sorted)
    else:
        print("Nenhum finding gerado.")
except Exception as e:
    print("findings_df indisponível:", e)

print("\nGate final (reuso de GATE_FAIL_ON):")
try:
    blocking = [f for f in findings if f['severity'] in GATE_FAIL_ON]
    if blocking:
        print(f"X {len(blocking)} problema(s) bloqueante(s):")
        for b in blocking:
            print('  -', b['rule'], '|', b['object_name'], '|', b.get('details'))
        # Se quiser que o notebook quebre quando houver bloqueios, descomente a linha abaixo
        # raise Exception('Validacao dinamica falhou: findings bloqueantes')
    else:
        print('OK - sem problemas bloqueantes')
except Exception as e:
    print("Não foi possível reavaliar o gate:", e)


## 15. Resultados consolidados

Esta seção resume, em um único lugar, os principais indicadores calculados nas seções anteriores (itens 1 a 12) e apresenta visualizações rápidas para facilitar a interpretação dos resultados:

- **Visão geral do health** (score, status, tamanho do modelo, número de findings)
- **Performance das queries DAX** (média, p95)
- **Complexidade das medidas**
- **Distribuição de severidade dos findings**

Use esta área como "painel" final da validação dinâmica antes de seguir para o deploy.


In [ ]:
# 15. Painel visual dos resultados (itens 1 a 13)

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("ggplot")

# IMPORTANTE: esta célula supõe que TODAS as seções 1–13 já foram executadas
# (especialmente a célula "14. Consolidação final dos resultados").

# Tamanho do modelo – visual em faixas (bom / atenção / crítico)
if pd.notnull(model_size):
    att = CONFIG.get('model_size_mb_attention', 512)
    crit = CONFIG.get('model_size_mb_critical', 1024)
    max_x = max(model_size * 1.1, crit * 1.2)

    fig, ax = plt.subplots(figsize=(5, 1.2))

    # Faixas de cor no eixo X
    ax.axvspan(0, att, color='#2ca02c', alpha=0.3, label=f'Bom (< {att} MB)')
    ax.axvspan(att, crit, color='#ff7f0e', alpha=0.3, label=f'Atenção ({att}–{crit} MB)')
    ax.axvspan(crit, max_x, color='#d62728', alpha=0.3, label=f'Crítico (>= {crit} MB)')

    # Linha vertical com o tamanho do modelo
    ax.axvline(model_size, color='#1f77b4', linewidth=3)
    ax.set_xlim(0, max_x)
    ax.set_yticks([])
    ax.set_xlabel('Tamanho do modelo (MB)')
    ax.set_title(f'Model size = {model_size:.1f} MB  |  Findings = {num_findings}')
    ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()

# --------------------------------------------------------------------
# Itens 5 e 10 – Performance das queries DAX (média e p95)
# --------------------------------------------------------------------
print("\n=== Itens 5 e 10 - Performance das queries DAX ===")

if 'perf_df' in globals() and isinstance(perf_df, pd.DataFrame) and len(perf_df):
    display(perf_df)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(perf_df['query'], perf_df['avg_ms'], color='#1f77b4')
    ax.axhline(y=CONFIG.get('perf_ms_attention', 1000), color='orange', linestyle='--', label='Atenção')
    ax.axhline(y=CONFIG.get('perf_ms_critical', 3000), color='red', linestyle='--', label='Crítico')
    ax.set_ylabel('Tempo médio (ms)')
    ax.set_title('Tempo médio por query DAX')
    ax.set_xticklabels(perf_df['query'], rotation=45, ha='right')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("perf_df não disponível – rode a seção 5 antes.")

print("\n--- Item 10 - Performance avançada (p95) ---")

if 'perf_percentiles_df' in globals() and isinstance(perf_percentiles_df, pd.DataFrame) and len(perf_percentiles_df):
    display(perf_percentiles_df)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(perf_percentiles_df['query'], perf_percentiles_df['p95_ms'], color='#9467bd')
    p95_limit = CONFIG.get('perf_ms_critical', 3000) * CONFIG.get('perf_p95_multiplier', 1.5)
    ax.axhline(y=p95_limit, color='red', linestyle='--', label=f'Limite p95 ({p95_limit:.0f} ms)')
    ax.set_ylabel('p95 (ms)')
    ax.set_title('Tempo p95 por query DAX')
    ax.set_xticklabels(perf_percentiles_df['query'], rotation=45, ha='right')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("perf_percentiles_df não disponível – rode a seção 10 antes.")

print("\n--- Item 10 - Teste concorrente ---")

if 'concurrent_df' in globals() and isinstance(concurrent_df, pd.DataFrame) and len(concurrent_df):
    display(concurrent_df)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(['single', 'concurrent'],
           [concurrent_df.iloc[0]['single_avg_ms'], concurrent_df.iloc[0]['concurrent_avg_ms']],
           color=['#1f77b4', '#ff7f0e'])
    ax.set_ylabel('Tempo médio (ms)')
    ax.set_title('Degradação sob carga')
    plt.tight_layout()
    plt.show()
else:
    print('Teste concorrente não executado ou sem resultados (seção 10).')

# --------------------------------------------------------------------
# Item 7 – Colunas (cardinalidade e compressão)
# --------------------------------------------------------------------
print("\n=== Item 7 - Colunas (cardinalidade e compressão) ===")

if 'cardinality_df' in globals() and isinstance(cardinality_df, pd.DataFrame) and len(cardinality_df):
    top_cols = cardinality_df.sort_values('Cardinality', ascending=False).head(10)
    display(top_cols)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh(top_cols['Table Name'] + '.' + top_cols['Column Name'], top_cols['Cardinality'], color='#17becf')
    ax.set_xlabel('Cardinalidade')
    ax.set_title('Colunas com maior cardinalidade (top 10)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("cardinality_df não disponível – rode a seção 7 antes.")

if 'compression_df' in globals() and isinstance(compression_df, pd.DataFrame) and len(compression_df):
    display(compression_df)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(compression_df['table'], compression_df['dict_data_ratio'], color='#8c564b')
    ax.axhline(CONFIG.get('compression_ratio_attention', 0.35), color='orange', linestyle='--', label='Atenção')
    ax.axhline(CONFIG.get('compression_ratio_critical', 0.60), color='red', linestyle='--', label='Crítico')
    ax.set_ylabel('Dict/Data ratio')
    ax.set_title('Compressão por tabela')
    ax.set_xticklabels(compression_df['table'], rotation=45, ha='right')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('compression_df não disponível – rode a seção 7 antes.')

# --------------------------------------------------------------------
# Item 8 – Tabelas e armazenamento (relacionamentos, segmentos)
# --------------------------------------------------------------------
print("\n=== Item 8 - Tabelas e armazenamento (relacionamentos e segmentos) ===")

if 'relationships_quality_df' in globals() and isinstance(relationships_quality_df, pd.DataFrame) and len(relationships_quality_df):
    display(relationships_quality_df.head(20))
    fig, ax = plt.subplots(figsize=(6, 4))
    top_rels = relationships_quality_df.sort_values('ratio', ascending=False).head(10)
    ax.barh(top_rels['from'] + ' -> ' + top_rels['to'], top_rels['ratio'], color='#ff7f0e')
    ax.set_xlabel('Proporção (lado-muitos / lado-um)')
    ax.set_title('Relacionamentos com maior proporção')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('relationships_quality_df não disponível – rode a seção 8 antes.')

if 'segments_df' in globals() and isinstance(segments_df, pd.DataFrame) and len(segments_df):
    display(segments_df.head(20))
    fig, ax = plt.subplots(figsize=(6, 4))
    top_seg = segments_df.sort_values('rows_per_segment', ascending=False).head(10)
    ax.barh(top_seg['table'], top_seg['rows_per_segment'], color='#1f77b4')
    ax.set_xlabel('Linhas por segmento')
    ax.set_title('Segmentos com mais linhas')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('segments_df não disponível – rode a seção 8 antes.')

# --------------------------------------------------------------------
# Item 9 – Complexidade das medidas DAX
# --------------------------------------------------------------------
print("\n=== Item 9 - Complexidade das medidas DAX (top 10) ===")

if 'measure_complexity_df' in globals() and isinstance(measure_complexity_df, pd.DataFrame) and len(measure_complexity_df):
    top_meas = measure_complexity_df.sort_values('complexity', ascending=False).head(10)
    display(top_meas)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh(top_meas['measure'], top_meas['complexity'], color='#8c564b')
    ax.set_xlabel('Score de complexidade')
    ax.set_title('Medidas mais complexas (top 10)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('measure_complexity_df não disponível – rode a seção 9 antes.')

# --------------------------------------------------------------------
# Item 11 – RLS e refresh
# --------------------------------------------------------------------
print("\n=== Item 11 - RLS e refresh ===")

if 'rls_df' in globals() and isinstance(rls_df, pd.DataFrame) and len(rls_df):
    display(rls_df)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(rls_df['role'], rls_df['overhead_x'], color='#d62728')
    ax.axhline(CONFIG.get('rls_overhead_attention', 1.5), color='orange', linestyle='--', label='Atenção')
    ax.axhline(CONFIG.get('rls_overhead_critical', 3.0), color='red', linestyle='--', label='Crítico')
    ax.set_ylabel('Overhead (x)')
    ax.set_title('Impacto de RLS por role')
    plt.tight_layout()
    plt.show()
else:
    print('rls_df não disponível ou RLS_TEST_ROLES vazio – seção 11.')

if 'refresh_df' in globals() and isinstance(refresh_df, pd.DataFrame) and len(refresh_df):
    display(refresh_df)
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.bar(refresh_df['table'], refresh_df['refresh_ms'], color='#1f77b4')
    ax.set_ylabel('Tempo de refresh (ms)')
    ax.set_title('Tempo de refresh por tabela (teste)')
    plt.tight_layout()
    plt.show()
else:
    print('refresh_df não disponível ou REFRESH_TEST_TABLE=None – seção 11.')

# --------------------------------------------------------------------
# Item 12 – Histórico de health
# --------------------------------------------------------------------
print("\n=== Item 12 - Histórico de health (quando WRITE_TO_LAKEHOUSE=True) ===")

if 'history_df' in globals() and isinstance(history_df, pd.DataFrame) and len(history_df):
    display(history_df)
    if 'health_score' in history_df.columns:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.plot(pd.to_datetime(history_df['run_ts']), history_df['health_score'], marker='o')
        ax.set_xlabel('run_ts')
        ax.set_ylabel('Health score')
        ax.set_title('Evolução histórica do health')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('history_df não disponível – WRITE_TO_LAKEHOUSE=False ou ainda sem histórico.')

print("\n=== Painel visual concluído ===")


## 16. Deploy para PRD

**O que faz:**

Esta etapa usa o **Deployment Pipeline do Fabric** para promover, de forma programática, os artefatos da workspace de origem (Dev/Test) para a workspace de destino (por exemplo, PRD). A célula de código abaixo chama a API oficial do Fabric (`deploymentPipelines/deploy`) para disparar o deploy entre as stages configuradas no pipeline.

**Como funciona internamente:**

1. Lê o `PIPELINE_ID` (ID do deployment pipeline) e os nomes das workspaces de origem e destino.
2. Consulta os **stages** do pipeline via API do Fabric e descobre qual `stageId` corresponde a cada workspace.
3. Monta a requisição de deploy com `sourceStageId` e `targetStageId`, incluindo uma nota de auditoria.
4. Envia a requisição `POST /v1/deploymentPipelines/{id}/deploy`.
5. Exibe um resumo da operação: HTTP status, `operation-id`, `deployment-id`, `Location` e `Retry-After`.

**Resultado esperado:**

- Em caso de sucesso (HTTP 200/202), o serviço aceita o deploy e inicia uma **operação de longa duração** (LRO) que copia e atualiza os artefatos configurados no pipeline da stage de origem para a stage de destino.
- Você pode usar os IDs exibidos (principalmente `operation-id` e `deployment-id`) para acompanhar o progresso e o status do deploy no Fabric (ou em automações externas).

> Observação: Esta etapa **não revalida o health**; ela pressupõe que o notebook já executou as análises anteriores e que você decidiu seguir com o deploy com base nesses resultados.


In [ ]:
# 15. Deploy via Deployment Pipeline (Fabric API)
# 
# Esta célula dispara o deploy da stage de origem para a stage de destino
# usando a API nova do Fabric (deploymentPipelines/deploy) e exibe um resumo
# dos principais identificadores para acompanhamento.
#
# Pré-requisitos:
# - PIPELINE_ID: GUID do deployment pipeline (da URL do Fabric)
# - SRC_WS_NAME: nome da workspace de origem (já vinculada à primeira stage)
# - DST_WS_NAME: nome da workspace de destino (já vinculada à segunda stage)

PIPELINE_ID = "<pipeline_id>"  # test-deployment-pipeline
SRC_WS_NAME = "<workspace_dev>"
DST_WS_NAME = "<workspace_prd>"

import requests

# Token para chamar a API do Fabric
pbi_token = notebookutils.credentials.getToken("pbi")
headers_fabric = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type": "application/json"
}

print("=== 15. Deploy via Deployment Pipeline (Fabric) ===")
print(f"Pipeline ID: {PIPELINE_ID}")
print(f"Origem (workspace): {SRC_WS_NAME}")
print(f"Destino (workspace): {DST_WS_NAME}")

# 1) Descobrir os stages do pipeline via Fabric API
stages_url = f"https://api.fabric.microsoft.com/v1/deploymentPipelines/{PIPELINE_ID}/stages"
resp_stages = requests.get(stages_url, headers=headers_fabric)
print("\n[1/3] Consultando stages do pipeline...")
print("Status stages:", resp_stages.status_code)

if not resp_stages.ok:
    print("Erro ao listar stages do pipeline (Fabric API):")
    print(resp_stages.text[:2000])
else:
    stages = resp_stages.json().get("value", [])
    if not stages:
        raise RuntimeError("Nenhuma stage retornada pela API do Fabric para este pipeline.")

    print("Stages encontrados:")
    for s in stages:
        print("- stageId =", s.get("id"), "| order =", s.get("order"), "| workspaceId =", s.get("workspaceId"))

    # 2) Resolver sourceStageId e targetStageId com base nos nomes das workspaces
    print("\n[2/3] Resolvendo sourceStageId e targetStageId...")

    # Descobrir IDs das workspaces via API clássica do Power BI
    groups_resp = requests.get("https://api.powerbi.com/v1.0/myorg/groups", headers=headers_fabric)
    if not groups_resp.ok:
        print("Erro ao listar groups (workspaces):")
        print(groups_resp.text[:2000])
        raise SystemExit()

    groups = groups_resp.json().get("value", [])
    ws_name_by_id = {g["id"]: g["name"] for g in groups}

    src_stage_id = dst_stage_id = None
    for s in stages:
        ws_id = s.get("workspaceId")
        ws_name = ws_name_by_id.get(ws_id, ws_id)
        if ws_name == SRC_WS_NAME:
            src_stage_id = s.get("id")
        if ws_name == DST_WS_NAME:
            dst_stage_id = s.get("id")

    print("sourceStageId:", src_stage_id)
    print("targetStageId:", dst_stage_id)

    if not src_stage_id or not dst_stage_id:
        raise RuntimeError("Não foi possível resolver sourceStageId/targetStageId a partir dos nomes das workspaces.")

    # 3) Chamar o deploy usando a API do Fabric
    deploy_url = f"https://api.fabric.microsoft.com/v1/deploymentPipelines/{PIPELINE_ID}/deploy"
    body = {
        "sourceStageId": src_stage_id,
        "targetStageId": dst_stage_id,
        "note": "Deploy disparado a partir do notebook de validação dinâmica"
        # Opcional: adicionar "items" para deploy seletivo
        # "items": [
        #     {"sourceItemId": "<GUID-semantic-model>", "itemType": "SemanticModel"},
        #     {"sourceItemId": "<GUID-report>",       "itemType": "Report"}
        # ]
    }

    print("\n[3/3] Disparando deploy (Fabric API)...")
    resp_deploy = requests.post(deploy_url, headers=headers_fabric, json=body)
    print("HTTP status:", resp_deploy.status_code)

    if resp_deploy.status_code in (200, 202):
        print("\nDeploy aceito pelo serviço.")
        # A API de Fabric usa Long Running Operations; cabeçalhos trazem IDs úteis
        loc = resp_deploy.headers.get("Location")
        op_id = resp_deploy.headers.get("x-ms-operation-id")
        dep_id = resp_deploy.headers.get("deployment-id")
        retry_after = resp_deploy.headers.get("Retry-After")

        print("Resumo da operação de deploy:")
        print("- Operation Id:", op_id)
        print("- Deployment Id:", dep_id)
        print("- Location (para acompanhamento):", loc)
        print("- Retry-After (sugestão de espera antes de consultar o status):", retry_after)
        print("\nVocê pode usar o Operation Id para consultar o status da operação no Fabric (get-operation-state / get-operation-result).")
    else:
        print("\nFalha ao iniciar deploy:")
        print(resp_deploy.text[:4000])
